<a href="https://www.kaggle.com/code/pavankumar960/supermarket-sales-analysis-kpis?scriptVersionId=240824723" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Business Objective

This analysis aims to understand sales, shipping efficiency, and profit trends across regions, product categories, and customer segments to help improve business decisions and profitability.


## Table of Contents

1. [Business Objective](#Busines-Objective)
2. [Importing Libraries](#Importing-Libraries)
3. [Data Loading](#Data-Loading)
4. [Data Preprocessing](#Data-Preprocessing)
5. [Data Cleaning](#Data-Cleaning)
6. [Feature Engineering](#Feature-Engineering)
7. [Exploratory Data Analysis (EDA)](#Exploratory-Data-Analysis-(EDA))
9. [Conclusion](#Conclusion)




# Importing Libraries

In [ ]:
import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import plotly.express as px

pd.set_option('display.float_format', '{:,.2f}'.format)

import warnings
warnings.filterwarnings("ignore")

# Data Loading

In [ ]:
df = pd.read_excel("/kaggle/input/supermarket-dataset/().xlsx", parse_dates=['Order Date','Ship Date'])

# Data Preprocessing

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
# Converting Object type to Float for Sales and Profit

for col in ['Profit','Sales']:
    df[col] = (
        df[col]
          .str.replace('[\$,]', '', regex=True)
          .astype(float)
    )

In [ ]:
df.describe()

# Data Cleaning

In [ ]:
# checking for Nulls

df.isnull().sum()

In [ ]:
# Checking for Duplicates

df[df.duplicated(keep=False)]

In [ ]:
df = df.drop_duplicates()

# Feature Engineering

In [ ]:
df['Profit per Unit'] = df['Profit'].astype(float) / df['Quantity']

In [ ]:
df['Sales per Unit'] = df['Sales'].astype(float) / df['Quantity']

In [ ]:
df['Has Discount'] = df['Discount'].apply(lambda x: 1 if x > 0 else 0)

In [ ]:
df['Profit Category'] = df['Profit'].astype(float).apply(lambda x: 'High Profit' if x > 100 else ('Loss' if x < 0 else 'Low Profit'))

In [ ]:
df['Order Month'] = df['Order Date'].dt.month
df['Order Year'] = df['Order Date'].dt.year

In [ ]:
df['Delivery Day'] = df['Ship Date'].dt.day_name()
df['Weekend Delivery'] = df['Delivery Day'].isin(['Saturday', 'Sunday'])

In [ ]:
df['Order Value'] = df['Sales'].astype(float) * (1 - df['Discount'])

In [ ]:
customer_counts = df['Customer Name'].value_counts()
df['Customer Frequency'] = df['Customer Name'].map(customer_counts)

In [ ]:
region_profit = df.groupby('Region')['Profit Ratio'].mean()
df['Region Profit Avg'] = df['Region'].map(region_profit)

In [ ]:
df.describe()

# Exploratory Data Analysis (EDA)

In [ ]:
# Key Performance Indicators

kpi = pd.DataFrame({
    'Total Sales': [df['Sales'].sum()],
    'Total Profit': [df['Profit'].sum()],
    'Overall Profit %': [df['Profit'].sum() / df['Sales'].sum() * 100],
    'Orders': [df['Order ID'].nunique()],
    'Customers': [df['Customer Name'].nunique()],
    'Products': [df['Product Name'].nunique()]
})
kpi.T.rename(columns={0:'Value'})


In [ ]:
# Total Sales by Category

plt.figure(figsize=(8,5))
sns.barplot(data=df, x='Category', y='Sales', estimator=sum, ci=None)
plt.title("Total Sales by Category")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Timeline of Monthly Sales and Profit

ts = (df.set_index('Order Date').resample('M').agg({'Sales':'sum','Profit':'sum'}))

fig, ax1 = plt.subplots(figsize=(10,5))
ts['Sales'].plot(ax=ax1)
ax2 = ax1.twinx()
ts['Profit'].plot(ax=ax2, linestyle='--')
ax1.set_title('Monthly Sales vs Profit')
ax1.set_ylabel('Sales')
ax2.set_ylabel('Profit')
plt.show()

In [ ]:
# profit by Region

region_profit = df.groupby('Region')['Profit'].sum().sort_values()

plt.figure(figsize=(8,5))
region_profit.plot(kind='barh', color='skyblue')
plt.title("Profit by Region")
plt.xlabel("Profit")
plt.tight_layout()
plt.show()


In [ ]:
# Sales by Region

region = (df.groupby('Region')
            .agg(Sales=('Sales','sum'),
                 Profit=('Profit','sum'),
                 Orders=('Order ID','nunique'))
            .sort_values('Sales', ascending=False))

region.plot(kind='bar', y=['Sales','Profit'], figsize=(8,4))
plt.title('Sales & Profit by Region')
plt.show()


In [ ]:
# Average Profit Ratio by Category

pivot = df.pivot_table(index='Sub-Category',
                       columns='Category',
                       values='Profit Ratio',
                       aggfunc='mean')

pivot = pivot.sort_values(by=list(pivot.columns), ascending=False)

plt.figure(figsize=(8,6))
sns.heatmap(pivot, annot=True, fmt='.1%', cmap='RdYlGn')
plt.title('Mean Profit Ratio by Category/Sub-Category')
plt.show()


In [ ]:
# Sales vs Profit Bubble Chart

fig = px.scatter(df, x='Sales', y='Profit', size='Quantity', color='Category',
                 hover_data=['Sub-Category', 'City', 'State'])
fig.update_layout(title="Sales vs Profit Bubble Chart")
fig.show()


In [ ]:
prod = (df.groupby('Product Name')
          .agg({'Sales':'sum','Profit':'sum'})
          .assign(abs_profit=lambda x: x['Profit'].abs())
          .sort_values('abs_profit', ascending=False)
          .head(10)
          .sort_values('Profit'))

prod[['Sales','Profit']].plot(kind='barh', figsize=(8,5))
plt.title('Top 10 Products by Profit')
plt.xlabel('Amount')
plt.show()


In [ ]:
df['Ship Lag'] = (df['Ship Date'] - df['Order Date']).dt.days
sns.boxplot(x='Ship Mode', y='Ship Lag', data=df)
plt.title('Shipping Lag by Mode')
plt.show()


In [ ]:
fig = px.sunburst(df,
                  path=['Region','Category','Sub-Category'],
                  values='Sales',
                  color='Profit',
                  color_continuous_scale='RdYlGn',
                  title='Sales hierarchy')
fig.show()


# Conclusion

- The dataset contains supermarket sales records from 2013 to 2016 in the United States.

- The overall profit margin is 12.46%.

- West and East regions contribute the most to the overall profit, while Central and South regions account for most of the losses, primarily from the Machines sub-category.

- Within the Furniture category, Chairs and Furnishings are profitable, whereas Bookcases and Tables result in losses.

- In Office Supplies, Labels, Paper, and Envelopes yield high profits, while Binders and Appliances underperform.

- For Technology, Copiers are highly profitable, while Machines lead to significant losses.